# Critic-Agent trên Google Colab — model open-source 14B chạy bằng Ollama trên GPU T4

Notebook chạy Critic-Agent với một model mã nguồn mở lớn hơn nhiều so với `qwen2.5:3b` trên Mac (máy 8 GB không chạy nổi model 7B trở lên), hoàn toàn miễn phí: Ollama chạy trên GPU T4 (16 GB) của Colab, Critic-Agent gọi nó qua `localhost`. Có hai chế độ (biến `MODE` ở ô cấu hình):

- **`"demo"`** — mở web Explorer có chế độ **LIVE** qua một đường link public tạm thời (Cloudflare tunnel). Vào link, chọn một bài trong danh sách, bấm **▶ Chạy LIVE** và xem từng agent làm việc; cuối lượt có **điểm bài báo 1–10** (Meta-reviewer, thang ICLR/NeurIPS) và các phản biện có trích dẫn. Mỗi lần chỉ chạy một bài.
- **`"calibrate"`** — chấm lần lượt mọi PDF trong thư mục và so với điểm tham chiếu trong `eval/calibration.json`.

**Cách dùng**
1. Nguồn bài báo (`PAPERS_SOURCE`): `"arxiv"` (mặc định) tự tải 11 bài trong `eval/calibration.json` từ arXiv, không cần Google Drive; `"drive"` dùng các PDF bạn tải lên `MyDrive/CO5151_Critic/papers`. `USE_DRIVE = True` lưu kết quả vào Drive để không mất khi Colab ngắt (cần cấp quyền Drive); `False` lưu trong phiên Colab.
2. **Runtime → Change runtime type → T4 GPU**.
3. Chọn `MODE`, rồi **Runtime → Run all**.
   - `demo`: ô *Demo LIVE* in ra đường link `https://….trycloudflare.com`. Ô đó tiếp tục chạy để giữ server sống; bấm **Stop** ở ô đó khi xong, rồi chạy ô *Xuất lên web* nếu muốn đưa các lượt đã chạy lên GitHub Pages.
   - `calibrate`: kết quả (`calibration_*.json`, `*.runs.jsonl`) lưu vào `<thư mục gốc>/results`. Nếu Colab bị ngắt, **Run all** lại: các bài đã chấm được bỏ qua.
4. Báo cáo từng lượt chạy nằm ở `<thư mục gốc>/runs/<run_id>/report.md` (lượt LIVE ở `runs/live/`); thư mục gốc là `MyDrive/CO5151_Critic` khi dùng Drive, `/content/critic` khi không.

**Lưu ý:** link tunnel là công khai (ai có link đều mở được) và chỉ sống khi notebook còn chạy; mỗi phiên Colab cho một link mới. Thời gian một bài: ô *Chạy thử một bài* (chế độ `calibrate`) đo thời gian thật; chưa đo trước trên T4.

**Chọn model (ô cấu hình):**
- `gpt-oss:20b` (mặc định, ~14 GB, model suy luận có thinking): có thể mạnh nhất vừa với T4 nhưng sát giới hạn bộ nhớ. Ô *Kiểm tra model* in ra `ollama ps`: cột PROCESSOR phải là `100% GPU`; nếu có phần CPU thì giảm `NUM_CTX` hoặc đổi model. `THINK` = `"low"`/`"medium"`/`"high"` (cao hơn = kỹ hơn nhưng chậm hơn).
- `qwen3:14b` (~9 GB, thinking bật/tắt được: `THINK = "false"` để nhanh) hoặc `qwen2.5:14b` (~9 GB, không có thinking, `THINK = ""`): an toàn về bộ nhớ.
- Model 24B trở lên không vừa 16 GB của T4.

In [ ]:
# Configuration
MODE = "demo"                  # "demo": live web page, pick one paper at a time; "calibrate": score every PDF
MODEL = "gpt-oss:20b"          # Ollama model tag; see the notes above
THINK = "medium"               # thinking models: "low" | "medium" | "high" (gpt-oss), "true"/"false" (qwen3); "" = not sent
JSON_FORMAT = True             # Ollama format="json"; set False if the model check below returns empty or broken JSON
NUM_CTX = 16384                # context window; check that `ollama ps` shows 100% GPU
SEEDS = "1"                    # calibrate mode: e.g. "1 2 3" to measure how stable the score is (3x the time)
PAPERS_SOURCE = "arxiv"        # "arxiv": download the papers of eval/calibration.json; "drive": PDFs in DRIVE_DIR/papers
USE_DRIVE = False              # True: keep runs and results on Google Drive (survive a disconnect; needs Drive access)
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/CO5151_Critic"

In [ ]:
# Check that a GPU is attached (should show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Where papers, runs and results live: Google Drive (survives a disconnect) or the session disk
if USE_DRIVE or PAPERS_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
BASE = DRIVE_DIR if USE_DRIVE else "/content/critic"
PAPERS_DIR = f"{DRIVE_DIR}/papers" if PAPERS_SOURCE == "drive" else f"{BASE}/papers"
RUNS_DIR = f"{BASE}/runs"
OUT_DIR = f"{BASE}/results"
!mkdir -p "$PAPERS_DIR" "$RUNS_DIR" "$OUT_DIR"
print("papers:", PAPERS_DIR, "| runs:", RUNS_DIR)

In [ ]:
# Install Ollama and start the server in the background (one request at a time: parallel requests would each
# reserve a full NUM_CTX cache on the GPU; the 5 parallel critics simply queue)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="1", OLLAMA_KEEP_ALIVE="-1", OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
# own session: interrupting a cell (SIGINT to the kernel's process group) must not stop the server
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/ollama.log").read()[-2000:])

In [ ]:
# Download the review model and the embedding model used by hybrid retrieval (a few minutes)
!ollama pull {MODEL} 2>&1 | tail -1
!ollama pull nomic-embed-text 2>&1 | tail -1
!ollama list

In [ ]:
# Model check: one JSON request like the ones Critic-Agent sends; prints the speed and whether the JSON parses
import json, requests
t = time.time()
body = {"model": MODEL, "stream": False,
        "messages": [{"role": "user", "content": "Give a 1-10 score for a paper that proposes RLHF for instruction "
                      'following. Return JSON only: {"overall": <1-10>, "justification": <one sentence>}'}],
        "options": {"temperature": 0, "num_ctx": NUM_CTX, "num_predict": 300 + (4096 if THINK not in ("", "false") else 0)}}
if JSON_FORMAT:
    body["format"] = "json"
if THINK:
    body["think"] = THINK if THINK in ("low", "medium", "high") else THINK == "true"
r = requests.post("http://localhost:11434/api/chat", timeout=1800, json=body).json()
print("thinking:", (r["message"].get("thinking") or "")[:300])
print("content:", r["message"]["content"])
print(json.loads(r["message"]["content"]))
print(f"{r.get('eval_count', 0) / (r.get('eval_duration', 1) / 1e9):.1f} tokens/s generation, "
      f"{time.time() - t:.0f}s total (first call includes loading the model onto the GPU)")
!ollama ps

In [ ]:
# Get the code and its pinned dependencies (uv installs Python 3.12 if the session has another version)
REPO = "/content/C05151_Next-generation_Software_Technology"
!if [ -d "$REPO" ]; then git -C "$REPO" fetch -q origin && git -C "$REPO" checkout -q "$BRANCH" && git -C "$REPO" pull -q; else git clone -q -b "$BRANCH" https://github.com/anho272202-HCMUT/C05151_Next-generation_Software_Technology.git "$REPO"; fi
%cd {REPO}
!git log --oneline -1
!pip install -q uv
!uv sync --frozen -q
# Offline tests with the mock LLM (run before .env exists, since .env would override the test settings)
!uv run pytest -q 2>&1 | tail -1

In [ ]:
# Papers: download them from arXiv (file names of eval/calibration.json), or use the PDFs uploaded to Drive
if PAPERS_SOURCE == "arxiv":
    !uv run python scripts/fetch_papers.py --out "$PAPERS_DIR"
!ls "$PAPERS_DIR"; echo "PDF count: $(ls "$PAPERS_DIR"/*.pdf 2>/dev/null | wc -l)"

In [ ]:
# Settings for a 14-20B model with a 16k context: larger evidence budgets than the 3B defaults (1800 tokens / 4 hits /
# 4000 chars), small enough that the longest prompt plus the thinking budget stays under NUM_CTX
settings = {
    "LLM_PROVIDER": "ollama", "LLM_MODEL": MODEL, "OLLAMA_BASE_URL": "http://localhost:11434",
    "EMBED_MODEL": "nomic-embed-text", "OLLAMA_NUM_CTX": NUM_CTX,
    "OLLAMA_THINK": THINK, "OLLAMA_THINK_TOKENS": 4096, "OLLAMA_JSON_FORMAT": int(JSON_FORMAT),
    "CRITIC_EVIDENCE_TOKENS": 4000, "CRITIC_SEARCH_K": 6, "CRITIC_FIRST_PAGES_CHARS": 10000,
    "CRITIC_GUARDRAIL": 1, "CRITIC_PAPER_SCORE": 1,
    "PAPERS_DIR": PAPERS_DIR, "RUNS_DIR": RUNS_DIR, "CACHE_DIR": "/content/critic_cache",
}
open(".env", "w").write("".join(f"{k}={v}\n" for k, v in settings.items()))
!cat .env

In [ ]:
# Calibrate mode: run one review first (the shortest PDF) to check the pipeline and measure the time per paper
import glob
if MODE == "calibrate":
    first = min(glob.glob(f"{PAPERS_DIR}/*.pdf"), key=os.path.getsize)
    print("Paper:", os.path.basename(first))
    t = time.time()
    !uv run critic-agent review "{os.path.basename(first)}" --seed 1 < /dev/null
    print(f"\n{(time.time() - t) / 60:.1f} min for one paper")

In [ ]:
# Calibrate mode: score every PDF. Rerun this cell (or Run all) after a disconnect: finished (paper, seed) pairs are skipped
if MODE == "calibrate":
    !uv run python scripts/calibrate.py --papers-dir "$PAPERS_DIR" --seeds $SEEDS --out-dir "$OUT_DIR" < /dev/null

In [ ]:
# Calibrate mode: results table; reports of every review are in RUNS_DIR/<run_id>/report.md
import pandas as pd
stem = "calibration_" + MODEL.replace(":", "-").replace("/", "-")
if MODE == "calibrate" and os.path.exists(f"{OUT_DIR}/{stem}.json"):
    summary = json.load(open(f"{OUT_DIR}/{stem}.json"))
    print({k: v for k, v in summary.items() if k != "rows"})
    display(pd.DataFrame(summary["rows"])[["file", "domain", "mean", "std", "scores", "reference"]])

## Demo LIVE (chế độ `demo`)
Ô dưới khởi động web Explorer kèm chế độ LIVE và mở một Cloudflare tunnel tạm thời (không cần tài khoản). Mở đường link in ra, vào tab chạy demo, chọn bài trong ô **Bài báo** rồi bấm **▶ Chạy LIVE**. Ô này chạy liên tục để giữ server; bấm **Stop** khi xong.

In [ ]:
# Live demo: web Explorer with live runs on port 8000, opened to the internet through a temporary Cloudflare tunnel
import re
if MODE == "demo":
    subprocess.run(f"git -C {REPO} pull -q", shell=True)  # pick up fixes pushed since the session started
    # rerunning this cell replaces the previous server and tunnel (each in its own session, so that
    # stopping the cell below does not kill them, nor the Ollama server)
    subprocess.run("pkill -f 'critic-agent serve'; pkill cloudflared", shell=True)
    server = subprocess.Popen(["uv", "run", "critic-agent", "serve", "--port", "8000"],
                              stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
                              start_new_session=True)
    for _ in range(120):
        try:
            print(urllib.request.urlopen("http://localhost:8000/api/health", timeout=2).read().decode())
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError(open("/content/server.log").read()[-2000:])
    subprocess.run("[ -x /usr/local/bin/cloudflared ] || wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/"
                   "latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                              stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT,
                              start_new_session=True)
    url = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m:
            url = m.group(0)
            break
        time.sleep(1)
    if not url:
        raise RuntimeError(open("/content/tunnel.log").read()[-2000:])
    print(f"\n>>> Demo LIVE: {url}\n    (link công khai, chỉ sống khi ô này còn chạy)\n")
    try:  # keep the cell (and the session) busy; show what the server is doing
        seen = 0
        while server.poll() is None and tunnel.poll() is None:
            time.sleep(30)
            runs = glob.glob(f"{RUNS_DIR}/live/*/output.json")
            if len(runs) != seen:
                seen = len(runs)
                print(time.strftime("%H:%M"), f"finished live runs: {seen}")
        print("server or tunnel stopped:", open("/content/server.log").read()[-1000:], open("/content/tunnel.log").read()[-1000:])
    except KeyboardInterrupt:
        print("Stopped. The live runs are in", f"{RUNS_DIR}/live")

## Xuất lên web (GitHub Pages)
Gom các lượt đã chạy (cả `calibrate` lẫn LIVE) thành dữ liệu cho web Explorer và nén vào `<thư mục gốc>/web_data.zip`. Giải nén file này vào thư mục `web/data/` của repo rồi push lên `main` thì GitHub Pages sẽ có các bài này để chọn và xem lại (kèm điểm bài báo).

In [ ]:
# Export the recorded runs for the web Explorer and zip web/data (to Drive when USE_DRIVE)
!uv run python scripts/export_web_data.py --runs-dir "$RUNS_DIR" --add
!cd web && zip -qr "$BASE/web_data.zip" data && ls -la "$BASE/web_data.zip"